## Off Policy Monte Carlo Control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from tqdm.notebook import trange
import matplotlib.colors as mcolors

In [ ]:
class GridworldEnv:
    def __init__(self, grid_size, holes=[]):
        self.grid_size = grid_size
        self.states = [(i, j) for i in range(grid_size) for j in range(grid_size)]
        self.terminal_states = [(0, 0), (grid_size-1, grid_size-1)]
        self.actions = ['up', 'down', 'left', 'right']
        self.holes = holes
        self.rewards = {s: -1 for s in self.states}
        self.rewards[(0, 0)] = 0
        self.rewards[(grid_size-1, grid_size-1)] = 1
        self.gamma = 0.9
        for hole in holes:
            self.rewards[hole] = -np.inf  # Holes have a very negative reward

    def is_terminal(self, state):
        return state in self.terminal_states

    def is_hole(self, state):
        return state in self.holes

    def get_next_state(self, state, action):
        if self.is_terminal(state) or self.is_hole(state):
            return state
        i, j = state
        if action == 'up':
            next_state = (max(i-1, 0), j)
        elif action == 'down':
            next_state = (min(i+1, self.grid_size-1), j)
        elif action == 'left':
            next_state = (i, max(j-1, 0))
        elif action == 'right':
            next_state = (i, min(j+1, self.grid_size-1))
        if self.is_hole(next_state):
            return state  # Stay in the same state if the next state is a hole
        return next_state

    def get_tprob_and_rewards(self, state, action):
        next_state = self.get_next_state(state, action)
        return [(1.0, next_state, self.rewards[next_state])]

    def display_grid(self, filename='gridworld_states.png'):
        fig, ax = plt.subplots()
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                state = (i, j)
                if state in self.terminal_states:
                    color = 'gray'
                elif state in self.holes:
                    color = 'black'
                else:
                    color = 'white'
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, edgecolor='black', facecolor=color))
                ax.text(j + 0.5, i + 0.5, f'({i},{j})', ha='center', va='center')
        ax.set_xlim(0, self.grid_size)
        ax.set_ylim(0, self.grid_size)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.title("Gridworld States")
        plt.gca().invert_yaxis()
        plt.savefig(filename, dpi=300)
        plt.show()

In [ ]:
# Initialize the environment
holes = [(1, 1), (2, 2)]
env = GridworldEnv(grid_size=6, holes=holes)

In [ ]:
env.display_grid()

### Monte Carlo Policy Evaluation

In [ ]:
def behaviour_policy(state):
    return random.choice(['up', 'down', 'left', 'right'])

def target_policy(Q, state):
    return max(env.actions, key=lambda a: Q[state][a])

def generate_episode(env, policy, max_count=100):
    episode = []
    state = random.choice(env.states)
    while env.is_terminal(state):
        state = random.choice(env.states)
    count = 0
    while not env.is_terminal(state):
        if count < max_count:
            action = policy(state)
            next_state = env.get_next_state(state, action)
            reward = env.rewards[next_state]
            episode.append((state, action, reward))
            state = next_state
            count += 1
        else:
            break
    return episode

def off_policy_mc_control(env, num_episodes, epsilon=0.1):
    Q = defaultdict(lambda: defaultdict(float))
    C = defaultdict(lambda: defaultdict(float))

    for _ in trange(num_episodes):
        episode = generate_episode(env, behaviour_policy)
        G = 0
        W = 1.0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = env.gamma * G + reward
            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])
            if action != target_policy(Q, state):
                break
            W /= 0.25
    
    # Extract the optimal policy from Q
    optimal_policy = {state: target_policy(Q, state) for state in env.states}
    return Q, optimal_policy

In [ ]:
# Run the algorithm
num_episodes = 5000000
Q, optimal_policy = off_policy_mc_control(env, num_episodes)

In [ ]:
def calculate_value_function(Q):
    V = defaultdict(float)
    for state in Q:
        V[state] = max(Q[state].values())
    return V

In [ ]:
V = calculate_value_function(Q)

In [ ]:
def plot_value_function(value_function, grid_size, title, filename='', cmap='viridis'):
    values = np.zeros((grid_size, grid_size))
    for (i, j), value in value_function.items():
        values[i, j] = value
    fig, ax = plt.subplots()
    cax = ax.matshow(values, cmap=cmap)
    text_color = 'white'
    if cmap == 'Greys':
        text_color = '0.5'
    for (i, j), value in value_function.items():
        ax.text(j, i, f"{value:.2f}", va='center', ha='center', color=text_color)
    fig.colorbar(cax)
    plt.title(title)
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_value_function(V, env.grid_size, 'Off-policy MC', 'gridworld_op_mcoff.png')

In [ ]:
plot_value_function(V, env.grid_size, 'Off-policy MC', 'gridworld_op_mcoff_bw.png', 'Greys')

In [ ]:
# Plotting the Optimal Policy
def plot_policy(policy, grid_size, filename=''):
    fig, ax = plt.subplots()
    for state, action in policy.items():
        if state in [(0, 0), (grid_size-1, grid_size-1)]:
            continue
        i, j = state
        if action == 'up':
            ax.arrow(j, grid_size-i-1, 0, 0.3, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'down':
            ax.arrow(j, grid_size-i-1, 0, -0.3, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'left':
            ax.arrow(j, grid_size-i-1, -0.3, 0, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'right':
            ax.arrow(j, grid_size-i-1, 0.3, 0, head_width=0.1, head_length=0.1, fc='k', ec='k')
    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(-0.5, grid_size-0.5)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels(range(grid_size))
    ax.set_yticklabels(range(grid_size-1, -1, -1))
    ax.grid(True)
    plt.title("Optimal Policy")
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_policy(optimal_policy, env.grid_size, 'gridworld_mcoff_op.png')

In [ ]:
def plot_q_function(q, grid_size, title, action, filename='', cmap='viridis'):
    values = np.zeros((grid_size, grid_size))
    for (i, j), value in q.items():
        values[i, j] = value[action]
    fig, ax = plt.subplots()
    cax = ax.matshow(values, cmap=cmap)
    text_color = 'white'
    if cmap == 'Greys':
        text_color = '0.5'
    for (i, j), value in q.items():
        ax.text(j, i, f"{value[action]:.2f}", va='center', ha='center', color=text_color)
    fig.colorbar(cax)
    plt.title(title)
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
# Plot the Q-function
plot_q_function(Q, env.grid_size, 'Action: up', 'up')

In [ ]:
plot_q_function(Q, env.grid_size, 'Action: down', 'down')

In [ ]:
plot_q_function(Q, env.grid_size, 'Action: left', 'left')

In [ ]:
plot_q_function(Q, env.grid_size, 'Action: right', 'right')

In [ ]:
optimal_policy